# Nanoplasma: export reduced H5 + plots 

This notebook has two workflows:

1) **Export reduced HDF5** (radial/spectra/time series) using your working exporter module `ana_extract_v2.py`.
2) **(Optional) Append map slices** (|E| and particle density maps) into the same H5 from raw openPMD fields/particles.

Everything is written/read from your scratch path:
`/p/scratch/jureap18/medina2/2026_nanoplasma/Run001_reduced.h5`


In [2]:
# --- Imports ---
import os, glob
import numpy as np
import h5py, glob, adios2
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import scipy.constants as sc
from nanoplasma_analysis_final import extract_step_from_filename

# IMPORTANT:
# Use your working exporter (no indentation issues).
# Ensure ana_extract_v2.py is in the SAME folder as this notebook.
from nanoplasma_analysis import NanoPlasmaRun


In [3]:
# --- User settings (EDIT HERE) ---\n",
#SIMOUTPUT = "/p/scratch/jureap18/medina2/2026_nanoplasma/001_OnceIonized/simOutput/"
#OUT_H5    = "/p/scratch/jureap18/medina2/2026_nanoplasma/Run001_reduced.h5"

#SIMOUTPUT = "/p/scratch/jureap18/medina2/2026_nanoplasma/003_Neutral_noAtomPhysics/simOutput/"
#OUT_H5    = "/p/scratch/jureap18/medina2/2026_nanoplasma/Run003_reduced.h5"

SIMOUTPUT = "/p/scratch/jureap18/medina2/2026_nanoplasma/007_He10e6_7e14Wcm-2/simOutput/"
OUT_H5    = "/p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5"

LASER_PEAK_STEP = 89603
ION_SPECIES = "He_i"
E_SPECIES   = "He_e"
ZMAX = 2

# Binning\n",
R_MAX_NM = 500.0
N_R = 2000
MU_BINS = 720
E_MAX_EV = 2000.0
N_E = 1200
LOG_E = True

## 1) Export reduced HDF5 (radial/spectra/time series)

In [5]:
run = NanoPlasmaRun(path=SIMOUTPUT, laser_peak_at_target=LASER_PEAK_STEP)
out = run.export_reduced_h5(
    out_h5=OUT_H5,
    store_slices=True,
    slice_every=1,          # or your cadence
    slice_y_nm=300.0,       # same as your plotting convention
    electron_density_field="He_e_all_density",
    ion_charge_density_field="He_i_all_density",
    store_pxy=True,
    p_bins=256,             # VMI resolution
    energy_binning="log"
    # p_max_e_SI=None,      # auto from E_max and mass
)
print("Wrote:", out)

#Default (linear KE, no change needed)
#run.export_reduced_h5(..., energy_binning="linear")

#Log binning (like the old scripts)
#run.export_reduced_h5(..., energy_binning="log", E_min_eV=1e-3, E_max_eV=2000, n_E=800)

#Piecewise (optional; good low-energy resolution without huge files)
#run.export_reduced_h5(...,energy_binning="piecewise", E_piecewise=[(0.0, 2.0, 0.02), (2.0, 50.0, 0.5), (50.0, 1000.0, 2)], E_max_eV=1000.0,)

/p/scratch/pwfa-trojan/medina2/conda_envs/picongpu/lib/python3.14/site-packages/numpy/_core/numeric.py:386: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


[export_reduced_h5] outputs: 14
[export_reduced_h5] r bins: 600 up to 2000.0 nm
[export_reduced_h5] mu bins: 360, energy bins: 800 up to 2000.0 eV
[export_reduced_h5] slices: on (14 stored)
[export] 1/14  step=0
[export] 2/14  step=10000
[export] 3/14  step=20000
[export] 4/14  step=30000
[export] 5/14  step=40000
[export] 6/14  step=50000
[export] 7/14  step=60000
[export] 8/14  step=70000
[export] 9/14  step=80000
[export] 10/14  step=90000
[export] 11/14  step=100000
[export] 12/14  step=110000
[export] 13/14  step=120000
[export] 14/14  step=130000
[export_reduced_h5] wrote: /p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5 (33.9 MB)
Wrote: /p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5


## 1b) Export Macrocells - reduced PIC position/momenta

**optional for better binning if necesary**

In [6]:

# --- settings ---
BLOCK = (8,8,8)        # macrocell block in PIC cells
STORE_EVERY = 1           # set to 5 or 10 if you want fewer timesteps
DTYPE = np.float32
COMPRESSION = "gzip"
COMP_LEVEL = 4

files = sorted(glob.glob(SIMOUTPUT + "/openPMD/*.bp5"))
if len(files) == 0:
    raise RuntimeError("No bp5 files found")

# Grid shape from first file (fields path!)
with adios2.Stream(files[0], "r") as f:
    step0 = extract_step_from_filename(files[0])
    for _ in f.steps():
        Nz, Ny, Nx = np.array(
            f.available_variables()[f"/data/{step0}/fields/B/x"]["Shape"].split(", "),
            dtype=int
        )
        break
print("Grid Nx,Ny,Nz =", Nx,Ny,Nz)

bx,by,bz = BLOCK
mx = (Nx + bx - 1)//bx
my = (Ny + by - 1)//by
mz = (Nz + bz - 1)//bz
nvox = mx*my*mz
print("Macro dims mx,my,mz =", mx,my,mz, "voxels =", nvox)

def lin(ix,iy,iz):
    return ix + mx*(iy + my*iz)

def bincount3(lidx, w, minlength):
    return np.bincount(lidx, weights=w, minlength=minlength)

with h5py.File(OUT_H5, "a") as h5:
    gm = h5.require_group("macrocell3d")
    gm.attrs["block_cells"] = np.array([bx,by,bz], dtype=np.int32)
    gm.attrs["macro_dims"] = np.array([mx,my,mz], dtype=np.int32)
    gm.attrs["grid_dims"]  = np.array([Nx,Ny,Nz], dtype=np.int32)
    gm.attrs["dtype"] = "float32"

    for k, fn in enumerate(files[::STORE_EVERY]):
        step = extract_step_from_filename(fn)
        print("macrocell step", step)

        with adios2.Stream(fn, "r") as f:
            for _ in f.steps():
                def read_part(f, step, sp):
                    """
                    Safe particle reader:
                    - returns empty arrays if the species/records are missing at this step
                    - avoids ADIOS2 ValueError for missing variables
                    """
                    # ask ADIOS2 what exists at this step
                    av = f.available_variables()
                
                    def read_or_empty(name, dtype=np.float64):
                        if name not in av:
                            return np.empty((0,), dtype=dtype)
                        return f.read(name).astype(dtype)
                
                    base = f"/data/{step}/particles/{sp}"
                    x  = read_or_empty(f"{base}/position/x", np.float64)
                    y  = read_or_empty(f"{base}/position/y", np.float64)
                    z  = read_or_empty(f"{base}/position/z", np.float64)
                    w  = read_or_empty(f"{base}/weighting",  np.float64)
                
                    # if no particles, bail early
                    if x.size == 0:
                        return x, y, z, w, np.empty((0,), np.float64), np.empty((0,), np.float64), np.empty((0,), np.float64)
                
                    px = read_or_empty(f"{base}/momentum/x", np.float64)
                    py = read_or_empty(f"{base}/momentum/y", np.float64)
                    pz = read_or_empty(f"{base}/momentum/z", np.float64)
                
                    return x, y, z, w, px, py, pz

                # electrons
                xe,ye,ze,we,pxe,pye,pze = read_part(f, step, E_SPECIES)
                mex = np.clip((np.floor(xe).astype(np.int64)//bx), 0, mx-1)
                mey = np.clip((np.floor(ye).astype(np.int64)//by), 0, my-1)
                mez = np.clip((np.floor(ze).astype(np.int64)//bz), 0, mz-1)
                le = lin(mex,mey,mez)

                # ions
                xi,yi,zi,wi,pxi,pyi,pzi = read_part(f, step,ION_SPECIES)
                mix = np.clip((np.floor(xi).astype(np.int64)//bx), 0, mx-1)
                miy = np.clip((np.floor(yi).astype(np.int64)//by), 0, my-1)
                miz = np.clip((np.floor(zi).astype(np.int64)//bz), 0, mz-1)
                li = lin(mix,miy,miz)

                # ion charge state Z from boundElectrons (if exists)
                try:
                    be = f.read(f"/data/{step}/particles/{ION_SPECIES}/boundElectrons").astype(np.int32)
                    wZ = f.read(f"/data/{step}/particles/{ION_SPECIES}/weighting").astype(np.float64)
                    Z = np.clip(ZMAX - be, 0, ZMAX).astype(np.float64)
                    qi = sc.e * Z * wZ
                except Exception:
                    qi = sc.e * wi  # fallback +1

                # charges
                qe = -sc.e * we

                # accumulate (1D), then reshape to (mz,my,mx) (z,y,x)
                qe_sum = bincount3(le, qe, nvox).reshape(mz,my,mx)
                qi_sum = bincount3(li, qi, nvox).reshape(mz,my,mx)

                Pex = bincount3(le, pxe*we, nvox).reshape(mz,my,mx)
                Pey = bincount3(le, pye*we, nvox).reshape(mz,my,mx)
                Pez = bincount3(le, pze*we, nvox).reshape(mz,my,mx)

                Pix = bincount3(li, pxi*wi, nvox).reshape(mz,my,mx)
                Piy = bincount3(li, pyi*wi, nvox).reshape(mz,my,mx)
                Piz = bincount3(li, pzi*wi, nvox).reshape(mz,my,mx)

                g = gm.require_group(f"step_{step:08d}")
                # overwrite if rerun
                for name in ["q_e_C","q_i_C","P_e_x","P_e_y","P_e_z","P_i_x","P_i_y","P_i_z"]:
                    if name in g: del g[name]

                g.create_dataset("q_e_C", data=qe_sum.astype(DTYPE), compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("q_i_C", data=qi_sum.astype(DTYPE), compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_e_x", data=Pex.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_e_y", data=Pey.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_e_z", data=Pez.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_i_x", data=Pix.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_i_y", data=Piy.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)
                g.create_dataset("P_i_z", data=Piz.astype(DTYPE),  compression=COMPRESSION, compression_opts=COMP_LEVEL, chunks=True)

                break

print("Done. Appended macrocell3d into:", OUT_H5)

Grid Nx,Ny,Nz = 1024 1600 1024
Macro dims mx,my,mz = 128 200 128 voxels = 3276800
macrocell step 0
macrocell step 10000
macrocell step 100000
macrocell step 110000
macrocell step 120000
macrocell step 130000
macrocell step 20000
macrocell step 30000
macrocell step 40000
macrocell step 50000
macrocell step 60000
macrocell step 70000
macrocell step 80000
macrocell step 90000
Done. Appended macrocell3d into: /p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5


##  Load reduced HDF5

In [6]:
h5 = h5py.File(OUT_H5, "r")

t_fs = h5["axes/time_fs"][:]
r_edges_nm = h5["axes/r_edges_m"][:] * 1e9
r_mid_nm   = h5["axes/r_mid_m"][:]   * 1e9

E_mid = h5["axes/E_mid_eV"][:]
mu_mid = h5["axes/mu_mid"][:]

ts = h5["timeseries"]
spec = h5["spectra"]
rad = h5["radial"]

print("Loaded:", OUT_H5)
print("Nt =", len(t_fs))

with h5py.File(h5,"r") as f:
    for k in f.keys():
        print(k, "->", list(f[k].keys())[:20])

Loaded: /p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5
Nt = 14


TypeError: expected str, bytes or os.PathLike object, not File

## Close H5

In [7]:
h5.close()
print("Closed:", OUT_H5)


Closed: /p/scratch/jureap18/medina2/2026_nanoplasma/Run007_reduced.h5
